# Day 08: SDE vs ODE Simulation in 2D

Welcome to practical 8! Compare deterministic Euler integration with Euler–Maruyama (simplified MIT Lab One).

This notebook follows the MIT lab style: abstract base classes, PyTorch, and LaTeX in markdown. Wherever you see a `# YOUR CODE HERE` marker, replace the `raise NotImplementedError(...)` below it with your own implementation.

In [ ]:
import math
from abc import ABC, abstractmethod
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

device = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

An ODE is $dX_t = u_t(X_t)\,dt$. An SDE adds noise: $dX_t = u_t(X_t)\,dt + \sigma_t\, dW_t$. Euler: $X_{t+h} = X_t + h\, u_t(X_t)$. Euler–Maruyama: $X_{t+h} = X_t + h\, u_t(X_t) + \sqrt{h}\,\sigma_t z$.

In [ ]:
class ODE(ABC):
    @abstractmethod
    def drift(self, x, t):
        pass

class SDE(ABC):
    @abstractmethod
    def drift(self, x, t):
        pass

    @abstractmethod
    def diffusion(self, x, t):
        pass

class RotatingODE(ODE):
    def drift(self, x, t):
        # YOUR CODE HERE — rotate velocity field
        raise NotImplementedError("TODO: complete this exercise")

class RotatingSDE(SDE):
    def drift(self, x, t):
        x1, x2 = x[:, 0:1], x[:, 1:2]
        return torch.cat([-x2, x1], dim=1)

    def diffusion(self, x, t):
        # YOUR CODE HERE
        raise NotImplementedError("TODO: complete this exercise")

### Exercise 8.1: Euler simulator

**Your job**: Implement `euler_step` for an ODE.

In [ ]:
def euler_simulate(ode, x0, steps=100):
    xs = [x0.clone()]
    x = x0.clone()
    dt = 1.0 / steps
    for i in range(steps):
        t = torch.tensor(i * dt)
        # YOUR CODE HERE
        raise NotImplementedError("TODO: complete this exercise")
    return torch.stack(xs, dim=0)

x0 = torch.tensor([[2.0, 0.0]])
traj = euler_simulate(RotatingODE(), x0)
plt.plot(traj[:, 0, 0].numpy(), traj[:, 0, 1].numpy())
plt.title("ODE trajectory (rotation)")
plt.xlabel("x1")
plt.ylabel("x2")
plt.axis("equal")
plt.show()

### Exercise 8.2: Euler–Maruyama

**Your job**: Implement stochastic integration.

In [ ]:
def euler_maruyama_simulate(sde, x0, steps=100, n_paths=200):
    dt = 1.0 / steps
    x = x0.repeat(n_paths, 1)
    for i in range(steps):
        t = torch.tensor(i * dt)
        # YOUR CODE HERE
        raise NotImplementedError("TODO: complete this exercise")
    return x

final = euler_maruyama_simulate(RotatingSDE(), torch.zeros(1, 2), steps=200, n_paths=500)
plt.scatter(final[:, 0].numpy(), final[:, 1].numpy(), s=8, alpha=0.5)
plt.title("SDE endpoints (cloud due to noise)")
plt.axis("equal")
plt.show()

### Exercise 8.3: Compare spread

**Your job**: Print std of final positions for ODE vs SDE.

In [ ]:
ode_final = euler_simulate(RotatingODE(), torch.zeros(1, 2), steps=200)[-1]
sde_final = euler_maruyama_simulate(RotatingSDE(), torch.zeros(1, 2), steps=200, n_paths=500)
print("ODE final (single path):", ode_final.numpy())
print("SDE final std:", sde_final.std(dim=0).numpy())

### Exercise 8.4: Vector field visualization

**Your job**: Plot drift arrows on a grid.

In [ ]:
grid = torch.linspace(-2, 2, 15)
X, Y = torch.meshgrid(grid, grid, indexing="xy")
pts = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=1)
ode = RotatingODE()
v = ode.drift(pts, torch.tensor(0.0))
plt.quiver(pts[:, 0], pts[:, 1], v[:, 0], v[:, 1], alpha=0.7)
plt.title("Drift field")
plt.axis("equal")
plt.show()

## Reflection (Day 8)

Take a few minutes to answer in your own words:

1. What was the most important concept you practiced today (ODEs vs SDEs)?
2. Where did you get stuck, and how did you resolve it?
3. How would you explain today's material to a classmate in two sentences?
4. What would you like to explore further?